# Consejo de Ministros — análisis integrado, una sola celda

**Vídeo:** `consejo_131026.mp4`, usado en `/workspace/equipo5/NOA/` por tu compañera.
Su prueba cubría 1355–1655 s. Este notebook analiza por defecto el vídeo completo.

Sube este notebook al servidor donde trabajabas y ejecuta la única celda de código.
El trabajo queda en segundo plano; mantén encendida la VM. Sigue el archivo `ejecucion.log`.
Primera ejecución: requiere internet para dependencias y modelos. No modifica tus notebooks anteriores.

Incluye YOLO11m, ByteTrack reiniciado por plano, PySceneDetect, OCR español/inglés,
tiempos reales verificados con ffprobe, CSV, base de datos reanudable y galería de recortes.
No elimina automáticamente tracks cortos, pequeños o inmóviles.

**Identidad:** los IDs son tracks locales. Revisa `revision.html` y une tracks en
`identidades_documentadas.csv` con un `persona_id` común y `confirmado=si`.
Las categorías hombre/mujer se introducen solo con una fuente documentada; no se infieren de la cara.
Vuelve a ejecutar la misma celda para regenerar agregados sin repetir planos ya completados.
No incluye análisis de audio. El texto OCR no se atribuye automáticamente a una persona.

La calidad debe validarse con anotaciones: no se garantiza identidad ni exactitud perfecta.
El muestreo de 10/s conserva el coste de la prueba previa; puedes cambiarlo a 30 antes de empezar.
Cambiar parámetros, vídeo, pesos o versiones requiere otra carpeta BASE.


In [1]:
# CONFIGURACIÓN: cambia únicamente estas líneas si hace falta.
from pathlib import Path
VIDEO = "/workspace/equipo5/NOA/consejo_131026.mp4"
BASE = Path("/workspace/equipo5/PAULA/consejo_integrado")
INICIO_S = 0.0
FIN_S = None             # vídeo completo; comparación exacta: INICIO_S=1355, FIN_S=1655
MUESTRAS_S = 10          # poner 30 para analizar todos los frames de este vídeo (más lento)
IMGSZ = 960             # resolución de entrada del detector; no garantiza más precisión
EXIGIR_GPU = True        # evita dejar accidentalmente una ejecución larga en CPU

# El resto funciona automáticamente. Puede ejecutarse desde notebook o como script.
import json, os, subprocess, sys, time
if sys.version_info < (3,10) or sys.version_info >= (3,14):
    raise RuntimeError("Usa Python 3.10–3.13 en tu servidor; el notebook original usa 3.13.")
video = Path(VIDEO).expanduser()
if not video.is_file():
    alternatives = [Path('/workspace/equipo5/NOA/videos/consejo_131026.mp4'),
                    Path('/workspace/equipo5/PAULA/consejo_131026.mp4'),
                    Path.cwd()/'consejo_131026.mp4']
    matches = [p for p in alternatives if p.is_file()]
    if len(matches)==1:
        video=matches[0]
    else:
        raise FileNotFoundError("No encuentro consejo_131026.mp4. Copia la ruta del vídeo completo a VIDEO. "
                                "Ruta usada por tu compañera: /workspace/equipo5/NOA/consejo_131026.mp4")
if MUESTRAS_S<=0 or INICIO_S<0 or (FIN_S is not None and FIN_S<=INICIO_S):
    raise ValueError('Revisa intervalo y frecuencia de muestreo.')
BASE=BASE.expanduser().resolve()
BASE.mkdir(parents=True,exist_ok=True)
pidfile=BASE/'proceso.json'
if pidfile.exists():
    pid=json.loads(pidfile.read_text())['pid']
    try:
        os.kill(pid,0)
    except ProcessLookupError:
        pass
    else:
        raise RuntimeError(f'El proceso {pid} sigue activo. Consulta {BASE / "ejecucion.log"}.')
(BASE/'modelos').mkdir(exist_ok=True)
cfg=dict(video=str(video.resolve()),salida=str(BASE/'resultados'),inicio_s=INICIO_S,fin_s=FIN_S,
         muestras_s=MUESTRAS_S,imgsz=IMGSZ,exigir_gpu=EXIGIR_GPU,
         modelo=str(BASE/'modelos/yolo11m.pt'),ocr_cada_s=.5,conf_ocr=.6)

WORKER = '"""Motor del notebook. Usar iniciar_consejo.py o la celda única del notebook."""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport html\nimport json\nimport math\nimport sqlite3\nimport subprocess\nimport sys\nimport time\nimport traceback\n\n\ndef atomic_json(path, value):\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef fingerprint(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef frame_times(video, fps, total):\n    """Preferir PTS reales; no convertir silenciosamente VFR en tiempo nominal."""\n    import shutil\n    if not shutil.which(\'ffprobe\'):\n        raise RuntimeError(\'Falta ffprobe (FFmpeg). Es necesario para verificar los tiempos del vídeo.\')\n    p = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'v:0\',\n                        \'-show_entries\', \'frame=best_effort_timestamp_time,pkt_duration_time\',\n                        \'-of\', \'json\', str(video)], capture_output=True, text=True, check=True)\n    records = json.loads(p.stdout)[\'frames\']\n    ts = [float(r[\'best_effort_timestamp_time\']) for r in records]\n    if not ts or any(not math.isfinite(x) for x in ts):\n        raise RuntimeError(\'Marcas temporales no válidas.\')\n    if any(b <= a for a, b in zip(ts, ts[1:])):\n        raise RuntimeError(\'PTS duplicados o no crecientes: normaliza el vídeo antes del análisis.\')\n    offset = ts[0]\n    ts = [t - offset for t in ts]\n    tail = float(records[-1].get(\'pkt_duration_time\', 0))\n    if tail <= 0:\n        tail = ts[-1] - ts[-2] if len(ts) > 1 else 1 / fps\n    if abs(len(ts) - total) > 1:\n        raise RuntimeError(f\'OpenCV declara {total} fotogramas y ffprobe encuentra {len(ts)}. Revisar vídeo.\')\n    return ts + [ts[-1] + tail]\n\n\ndef selected_frames(start, end, timestamps, rate):\n    """Muestrear por tiempo real, incluyendo el primer frame de cada plano."""\n    chosen, target = [], timestamps[start]\n    for i in range(start, end):\n        if timestamps[i] + 1e-8 >= target:\n            chosen.append(i)\n            target = timestamps[i] + 1 / rate\n    return chosen\n\n\ndef init_db(db):\n    db.executescript(\'\'\'\n    CREATE TABLE IF NOT EXISTS done(scene INTEGER PRIMARY KEY, seconds REAL);\n    CREATE TABLE IF NOT EXISTS frames(\n      scene INTEGER, frame INTEGER PRIMARY KEY, t REAL, dt REAL,\n      n INTEGER, text TEXT, ocr_t REAL);\n    CREATE TABLE IF NOT EXISTS det(\n      scene INTEGER, frame INTEGER, track TEXT, t REAL, dt REAL,\n      conf REAL, x1 REAL, y1 REAL, x2 REAL, y2 REAL,\n      area REAL, centrality REAL, border INTEGER,\n      PRIMARY KEY(frame, track));\n    CREATE TABLE IF NOT EXISTS ocr(\n      scene INTEGER, frame INTEGER, t REAL, blocks TEXT,\n      PRIMARY KEY(frame));\n    CREATE TABLE IF NOT EXISTS crops(\n      track TEXT, frame INTEGER, t REAL, path TEXT,\n      PRIMARY KEY(track, frame));\n    \'\'\')\n    db.commit()\n\n\ndef export_results(db, out, cfg, duration):\n    import pandas as pd\n    frames = pd.read_sql_query(\'SELECT * FROM frames ORDER BY frame\', db)\n    det = pd.read_sql_query(\'SELECT * FROM det ORDER BY frame,track\', db)\n    ocr = pd.read_sql_query(\'SELECT * FROM ocr ORDER BY frame\', db)\n    crops = pd.read_sql_query(\'SELECT * FROM crops ORDER BY track,frame\', db)\n    frames.to_csv(out / \'linea_tiempo.csv\', index=False)\n    det.to_csv(out / \'detecciones.csv\', index=False)\n    ocr.to_csv(out / \'ocr_crudo.csv\', index=False)\n    crops.to_csv(out / \'recortes.csv\', index=False)\n    rows = []\n    for tid, group in det.groupby(\'track\'):\n        rows.append(dict(track_id=tid, plano=int(group.scene.iloc[0]),\n                         inicio_s=group.t.min(), fin_intervalo_s=(group.t + group.dt).max(),\n                         muestras=len(group), tiempo_detectado_s=group.dt.sum(),\n                         area_media_pct=group.area.mean(), centralidad_horizontal=group.centrality.mean(),\n                         fraccion_borde=group.border.mean(), confianza_media=group.conf.mean(),\n                         revisar_corto=group.dt.sum() < 0.5,\n                         fraccion_area_menor_1pct=(group.area < 1).mean(),\n                         tiempo_area_mayor_8pct_s=group.loc[group.area > 8, \'dt\'].sum()))\n    summary = pd.DataFrame(rows, columns=[\'track_id\',\'plano\',\'inicio_s\',\'fin_intervalo_s\',\n        \'muestras\',\'tiempo_detectado_s\',\'area_media_pct\',\'centralidad_horizontal\',\n        \'fraccion_borde\',\'confianza_media\',\'revisar_corto\',\'fraccion_area_menor_1pct\',\n        \'tiempo_area_mayor_8pct_s\'])\n    summary.to_csv(out / \'resumen_tracks.csv\', index=False)\n    # Etiquetas humanas: nunca deducir categorías sensibles de la apariencia.\n    label_path = out / \'identidades_documentadas.csv\'\n    label_cols = [\'track_id\',\'persona_id\',\'categoria_documentada\',\'fuente\',\'tipo_presencia\',\'confirmado\']\n    labels = pd.read_csv(label_path, dtype=str, keep_default_na=False) if label_path.exists() else pd.DataFrame(columns=label_cols)\n    if any(c not in labels for c in label_cols) or labels.track_id.duplicated().any():\n        raise ValueError(\'La tabla de identidades tiene columnas ausentes o tracks duplicados.\')\n    missing = sorted(set(summary.track_id) - set(labels.track_id))\n    extra = pd.DataFrame([dict(zip(label_cols, [tid, \'\', \'no_documentado\', \'\', \'\', \'no\'])) for tid in missing], columns=label_cols)\n    labels = pd.concat([labels, extra], ignore_index=True)\n    labels.to_csv(label_path, index=False)\n    confirmed = labels[(labels.confirmado.str.lower() == \'si\') & (labels.persona_id != \'\')].copy()\n    bad = confirmed.groupby(\'persona_id\').categoria_documentada.nunique()\n    if (bad > 1).any():\n        raise ValueError(\'Una persona tiene categorías documentadas incompatibles. Revisa la tabla.\')\n    if ((confirmed.categoria_documentada != \'no_documentado\') & (confirmed.fuente == \'\')).any():\n        raise ValueError(\'Las categorías documentadas requieren una fuente.\')\n    merged = det.merge(confirmed, left_on=\'track\', right_on=\'track_id\')\n    # Una persona duplicada en pantalla se cuenta una vez por instante.\n    unique = merged.drop_duplicates([\'persona_id\',\'frame\'])\n    person_rows = []\n    for pid, g in unique.groupby(\'persona_id\'):\n        person_rows.append({\'persona_id\':pid, \'categoria_documentada\':g.categoria_documentada.iloc[0],\n                            \'tiempo_detectado_s\':g.dt.sum(), \'n_tracks\':merged.loc[merged.persona_id == pid,\'track\'].nunique()})\n    pd.DataFrame(person_rows, columns=[\'persona_id\',\'categoria_documentada\',\'tiempo_detectado_s\',\'n_tracks\']).to_csv(out / \'personas_confirmadas.csv\', index=False)\n    category = unique.groupby([\'frame\',\'categoria_documentada\']).persona_id.nunique().unstack(fill_value=0)\n    category = category.reindex(frames.frame, fill_value=0)\n    category.index.name = \'frame\'\n    category.to_csv(out / \'conteos_categorias_documentadas.csv\')\n    # Eventos de texto: solo igualdad normalizada; nunca atravesar un corte.\n    events = []\n    for r in frames.itertuples():\n        txt = \' \'.join(str(r.text).split())\n        if not txt:\n            continue\n        if events and events[-1][\'plano\'] == r.scene and events[-1][\'texto\'].casefold() == txt.casefold() and abs(events[-1][\'fin_s\'] - r.t) < 1e-5:\n            events[-1][\'fin_s\'] = r.t + r.dt\n        else:\n            events.append(dict(plano=r.scene, inicio_s=r.t, fin_s=r.t + r.dt, texto=txt))\n    pd.DataFrame(events, columns=[\'plano\',\'inicio_s\',\'fin_s\',\'texto\']).to_csv(out / \'eventos_texto.csv\', index=False)\n    cards = []\n    for row in summary.itertuples():\n        pics = crops[crops.track == row.track_id]\n        images = \'\'.join(f\'<figure><img loading="lazy" src="{html.escape(p.path, quote=True)}"><figcaption>{p.t:.2f} s</figcaption></figure>\' for p in pics.itertuples())\n        cards.append(f\'<article><h2>{html.escape(row.track_id)}</h2><p>Plano {row.plano}; {row.tiempo_detectado_s:.2f} s detectados; revisar corto: {row.revisar_corto}</p>{images}</article>\')\n    (out / \'revision.html\').write_text(\'<!doctype html><meta charset="utf-8"><title>Revisión de tracks</title><style>body{font:16px system-ui;margin:32px}article{border-top:1px solid #bbb;padding:16px}figure{display:inline-block;margin:8px}img{max-height:240px;max-width:240px}</style><h1>Tracks pendientes de revisión</h1><p>Un track no es una identidad confirmada. Revisa cortes, fragmentaciones, fotografías y ventanas superpuestas. Anota persona_id en identidades_documentadas.csv y vuelve a ejecutar para actualizar los resúmenes.</p>\' + \'\'.join(cards), encoding=\'utf-8\')\n    total_observed = float(frames.dt.sum()) if len(frames) else 0\n    report = dict(estado=\'completo\', duracion_intervalo_s=duration, tiempo_muestreado_s=total_observed,\n                  tiempo_con_tracks_s=float(frames.loc[frames.n > 0,\'dt\'].sum()), tracks=len(summary),\n                  personas_confirmadas=len(person_rows), detecciones=len(det), muestras_ocr=len(ocr),\n                  calidad_validada=False, categorias=\'Solo etiquetas documentadas; no inferidas de imágenes\',\n                  identidad=\'Tracks por plano; unión entre planos requiere confirmación\',\n                  medicion=\'Cada muestra representa hasta la siguiente muestra o el fin del plano. No interpola detecciones perdidas.\',\n                  audio=\'No se analiza audio ni se atribuye quién habla\')\n    atomic_json(out / \'resumen.json\', report)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n\n\ndef run(cfg):\n    import bisect\n    import importlib.metadata as metadata\n    import cv2\n    import easyocr\n    import torch\n    import yaml\n    from ultralytics import YOLO\n    from scenedetect import open_video, SceneManager, ContentDetector, ThresholdDetector\n    video = Path(cfg[\'video\']).expanduser().resolve()\n    if not video.is_file():\n        raise FileNotFoundError(f\'No existe el vídeo: {video}\')\n    out = Path(cfg[\'salida\']).expanduser().resolve()\n    out.mkdir(parents=True, exist_ok=True)\n    (out / \'recortes\').mkdir(exist_ok=True)\n    versions = {p:metadata.version(p) for p in [\'ultralytics\',\'scenedetect\',\'easyocr\',\'torch\',\'torchvision\',\'numpy\',\'pandas\']}\n    stamp = dict(config=cfg, video_sha256=fingerprint(video), versiones=versions, motor_sha256=fingerprint(Path(__file__)))\n    manifest = out / \'ejecucion.json\'\n    if manifest.exists() and json.loads(manifest.read_text(encoding=\'utf-8\')) != stamp:\n        raise RuntimeError(\'Han cambiado vídeo, configuración, motor o versiones. Usa otra carpeta SALIDA para no mezclar resultados.\')\n    atomic_json(manifest, stamp)\n    subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'], stdout=(out / \'entorno.txt\').open(\'w\', encoding=\'utf-8\'), check=True)\n    cap = cv2.VideoCapture(str(video))\n    if not cap.isOpened():\n        raise RuntimeError(\'OpenCV no puede abrir el vídeo.\')\n    fps, total = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    cap.release()\n    if not math.isfinite(fps) or fps <= 0 or total <= 0:\n        raise RuntimeError(\'FPS o número de fotogramas no válidos.\')\n    times_file = out / \'tiempos.json\'\n    if times_file.exists():\n        timestamps = json.loads(times_file.read_text())\n    else:\n        print(\'Verificando tiempos reales del vídeo...\', flush=True)\n        timestamps = frame_times(video, fps, total)\n        atomic_json(times_file, timestamps)\n    total = len(timestamps) - 1\n    lo = bisect.bisect_left(timestamps, cfg[\'inicio_s\'])\n    hi = min(total, bisect.bisect_left(timestamps, cfg[\'fin_s\'])) if cfg[\'fin_s\'] is not None else total\n    if not 0 <= lo < hi <= total:\n        raise ValueError(\'Intervalo fuera del vídeo o vacío.\')\n    scene_file = out / \'planos.json\'\n    if scene_file.exists():\n        scenes = json.loads(scene_file.read_text())\n    else:\n        print(\'Detectando cortes de plano (primera pasada)...\', flush=True)\n        manager = SceneManager()\n        manager.add_detector(ContentDetector(threshold=27, min_scene_len=max(1,round(fps * .3))))\n        manager.add_detector(ThresholdDetector(threshold=12, min_scene_len=max(1,round(fps * .3))))\n        source = open_video(str(video))\n        manager.detect_scenes(source, show_progress=True)\n        raw = manager.get_scene_list(start_in_scene=True)\n        # Las escenas de PySceneDetect tienen final exclusivo.\n        bounds = sorted(set([0,total] + [a.get_frames() for a,b in raw] + [b.get_frames() for a,b in raw]))\n        scenes = [[max(a,lo), min(b,hi)] for a,b in zip(bounds,bounds[1:]) if max(a,lo) < min(b,hi)]\n        atomic_json(scene_file, scenes)\n    assert scenes[0][0] == lo and scenes[-1][1] == hi\n    assert all(a[1] == b[0] for a,b in zip(scenes,scenes[1:]))\n    with (out / \'planos.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        w = csv.writer(f); w.writerow([\'plano\',\'frame_ini\',\'frame_fin_exclusivo\',\'inicio_s\',\'fin_s\'])\n        w.writerows((i,a,b,timestamps[a],timestamps[b]) for i,(a,b) in enumerate(scenes))\n    gpu = torch.cuda.is_available()\n    if cfg.get(\'exigir_gpu\', True) and not gpu:\n        raise RuntimeError(\'No hay GPU CUDA disponible. Para aceptar ejecución lenta pon EXIGIR_GPU=False.\')\n    print(\'Dispositivo:\', torch.cuda.get_device_name(0) if gpu else \'CPU\', flush=True)\n    model = YOLO(cfg[\'modelo\'])\n    model_file = Path(getattr(model,\'ckpt_path\',cfg[\'modelo\']))\n    if model_file.is_file():\n        weights = {\'path\':str(model_file.resolve()),\'sha256\':fingerprint(model_file)}\n        old_weights = out / \'pesos.json\'\n        if old_weights.exists() and json.loads(old_weights.read_text())[\'sha256\'] != weights[\'sha256\']:\n            raise RuntimeError(\'Los pesos han cambiado; utiliza otra carpeta de salida.\')\n        atomic_json(old_weights, weights)\n    reader = easyocr.Reader([\'es\',\'en\'], gpu=gpu, model_storage_directory=str(out.parent / \'modelos_easyocr\'))\n    tracker = out / \'bytetrack.yaml\'\n    tracker.write_text(yaml.safe_dump(dict(tracker_type=\'bytetrack\', track_high_thresh=.25,\n        track_low_thresh=.1, new_track_thresh=.4, track_buffer=max(1,round(2*min(fps,cfg[\'muestras_s\']))),\n        match_thresh=.8, fuse_score=True)), encoding=\'utf-8\')\n    db = sqlite3.connect(out / \'analisis.sqlite\')\n    init_db(db)\n    finished = {r[0] for r in db.execute(\'SELECT scene FROM done\')}\n    cap = cv2.VideoCapture(str(video))\n    try:\n        for sid,(start,end) in enumerate(scenes):\n            if sid in finished:\n                continue\n            started = time.monotonic()\n            last_notice = started\n            print(f\'Plano {sid+1}/{len(scenes)}: {timestamps[start]:.2f}–{timestamps[end]:.2f} s\', flush=True)\n            if getattr(model,\'predictor\',None) is not None:\n                for tr in getattr(model.predictor,\'trackers\',[]):\n                    tr.reset()\n            selected = selected_frames(start,end,timestamps,cfg[\'muestras_s\'])\n            durations = {f:timestamps[selected[j+1] if j+1 < len(selected) else end]-timestamps[f] for j,f in enumerate(selected)}\n            next_ocr, last_text, last_ocr = -1., \'\', None\n            crop_counts, last_crop = {}, {}\n            cap.set(cv2.CAP_PROP_POS_FRAMES, start)\n            pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))\n            if pos != start:\n                raise RuntimeError(f\'No se pudo posicionar el vídeo en {start} (posición {pos}).\')\n            # Una transacción por plano: un fallo revierte el plano incompleto.\n            with db:\n                wanted = set(selected)\n                for fi in range(start,end):\n                    if not cap.grab():\n                        raise RuntimeError(f\'Lectura interrumpida en fotograma {fi}; plano no confirmado.\')\n                    if fi not in wanted:\n                        continue\n                    ok, frame = cap.retrieve()\n                    if not ok:\n                        raise RuntimeError(f\'No se pudo decodificar el fotograma {fi}.\')\n                    t, dt = timestamps[fi], durations[fi]\n                    h,w = frame.shape[:2]\n                    result = model.track(frame, persist=True, tracker=str(tracker), classes=[0],\n                        conf=.1, iou=.7, imgsz=cfg[\'imgsz\'], device=0 if gpu else \'cpu\', verbose=False)[0]\n                    count = 0\n                    if result.boxes is not None and result.boxes.id is not None:\n                        for box, ident, conf in zip(result.boxes.xyxy.cpu().tolist(), result.boxes.id.int().cpu().tolist(), result.boxes.conf.cpu().tolist()):\n                            x1,y1,x2,y2 = box\n                            x1,x2 = max(0.,x1),min(float(w),x2)\n                            y1,y2 = max(0.,y1),min(float(h),y2)\n                            if x2 <= x1 or y2 <= y1:\n                                continue\n                            tid = f\'p{sid:05d}_t{ident:05d}\'\n                            area = 100*(x2-x1)*(y2-y1)/(w*h)\n                            cent = 1-2*abs((x1+x2)/(2*w)-.5)\n                            border = int(x1<=3 or y1<=3 or x2>=w-3 or y2>=h-3)\n                            db.execute(\'INSERT INTO det VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)\',\n                                       (sid,fi,tid,t,dt,float(conf),x1,y1,x2,y2,area,cent,border))\n                            count += 1\n                            if crop_counts.get(tid,0)<4 and t-last_crop.get(tid,-1e9)>=2:\n                                crop = frame[int(y1):math.ceil(y2),int(x1):math.ceil(x2)]\n                                if crop.size:\n                                    rel = f\'recortes/{tid}_{fi}.jpg\'\n                                    if not cv2.imwrite(str(out / rel),crop):\n                                        raise RuntimeError(\'No se pudo guardar un recorte.\')\n                                    db.execute(\'INSERT INTO crops VALUES (?,?,?,?)\',(tid,fi,t,rel))\n                                    crop_counts[tid] = crop_counts.get(tid,0)+1\n                                    last_crop[tid] = t\n                    if t + 1e-8 >= next_ocr:\n                        blocks = []\n                        for polygon,txt,score in reader.readtext(frame, detail=1, paragraph=False):\n                            blocks.append(dict(texto=txt,confianza=float(score),\n                                               poligono=[[float(x),float(y)] for x,y in polygon]))\n                        blocks.sort(key=lambda x:(min(p[1] for p in x[\'poligono\'])//20,min(p[0] for p in x[\'poligono\'])))\n                        db.execute(\'INSERT INTO ocr VALUES (?,?,?,?)\',(sid,fi,t,json.dumps(blocks,ensure_ascii=False)))\n                        last_text = \' | \'.join(b[\'texto\'] for b in blocks if b[\'confianza\']>=cfg[\'conf_ocr\'])\n                        last_ocr = t\n                        next_ocr = t + cfg[\'ocr_cada_s\']\n                    db.execute(\'INSERT INTO frames VALUES (?,?,?,?,?,?,?)\',(sid,fi,t,dt,count,last_text,last_ocr))\n                    if time.monotonic() - last_notice >= 30:\n                        print(f\'  Plano {sid+1}: segundo {t:.1f} / {timestamps[end]:.1f}\', flush=True)\n                        last_notice = time.monotonic()\n                db.execute(\'INSERT INTO done VALUES (?,?)\',(sid,time.monotonic()-started))\n            atomic_json(out / \'estado.json\', dict(estado=\'en_proceso\',planos_terminados=len(finished)+1,total_planos=len(scenes),ultimo_segundo=timestamps[end]))\n            finished.add(sid)\n            print(f\'Guardado plano {sid+1}; {time.monotonic()-started:.1f} s de cálculo\', flush=True)\n        export_results(db,out,cfg,timestamps[hi]-timestamps[lo])\n        atomic_json(out / \'estado.json\',dict(estado=\'completo\',planos_terminados=len(finished),total_planos=len(scenes)))\n    finally:\n        cap.release()\n        db.close()\n    print(\'TERMINADO. Resultados:\',out,flush=True)\n\n\nif __name__ == \'__main__\':\n    config = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    try:\n        run(config)\n    except BaseException as exc:\n        folder = Path(config[\'salida\'])\n        folder.mkdir(parents=True,exist_ok=True)\n        atomic_json(folder / \'error_ultimo_intento.json\',dict(error=type(exc).__name__,mensaje=str(exc),detalle=traceback.format_exc()))\n        atomic_json(folder / \'estado.json\',dict(estado=\'error\',mensaje=str(exc)))\n        raise\n'
BOOTSTRAP = "from pathlib import Path\nimport os, sys, subprocess, json, traceback\nroot=Path(__file__).resolve().parent\nlock=(root/'proceso.lock').open('a+')\ntry:\n    if os.name=='nt':\n        import msvcrt\n        lock.seek(0); lock.write('0'); lock.flush(); lock.seek(0)\n        msvcrt.locking(lock.fileno(),msvcrt.LK_NBLCK,1)\n    else:\n        import fcntl\n        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\nexcept OSError:\n    raise RuntimeError('Ya hay un análisis ejecutándose en esta carpeta.')\ntry:\n    (root/'FALLO.txt').unlink(missing_ok=True)\n    env=root/'entorno'\n    python=env/('Scripts/python.exe' if os.name=='nt' else 'bin/python')\n    if not python.exists():\n        subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(env)],check=True)\n    ready=env/'preparado.json'\n    requirements=['ultralytics>=8.3.0,<9','scenedetect==0.6.6','easyocr==1.7.2','lap>=0.5.12','pandas>=2,<4','pyyaml>=6,<7']\n    if not ready.exists():\n        print('Preparando entorno separado. La primera vez requiere internet.',flush=True)\n        subprocess.run([str(python),'-m','pip','install','--disable-pip-version-check',*requirements],check=True)\n        ready.write_text(json.dumps(requirements))\n    subprocess.run([str(python),'-u',str(root/'analizar_consejo.py'),str(root/'config.json')],check=True)\nexcept BaseException:\n    (root/'FALLO.txt').write_text(traceback.format_exc(),encoding='utf-8')\n    raise\nfinally:\n    pidfile=root/'proceso.json'\n    if pidfile.exists():\n        try:\n            if json.loads(pidfile.read_text())['pid']==os.getpid():\n                pidfile.unlink()\n        except (OSError,ValueError,KeyError):\n            pass\n    lock.close()\n"

(BASE/'analizar_consejo.py').write_text(WORKER,encoding='utf-8')
(BASE/'preparar_y_ejecutar.py').write_text(BOOTSTRAP,encoding='utf-8')
(BASE/'config.json').write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
logpath=BASE/'ejecucion.log'
options={'start_new_session':True} if os.name!='nt' else {'creationflags':subprocess.CREATE_NO_WINDOW}
with logpath.open('a',encoding='utf-8') as log:
    log.write('\nNUEVO INTENTO '+time.strftime('%Y-%m-%d %H:%M:%S')+'\n');log.flush()
    process=subprocess.Popen([sys.executable,'-u',str(BASE/'preparar_y_ejecutar.py')],
                             cwd=str(BASE),stdin=subprocess.DEVNULL,stdout=log,stderr=subprocess.STDOUT,**options)
pidfile.write_text(json.dumps({'pid':process.pid,'video':str(video),'inicio':time.time()}),encoding='utf-8')
time.sleep(2)
if process.poll() is not None:
    raise RuntimeError('El proceso terminó al arrancar. Mira '+str(logpath))
print('LANZADO EN SEGUNDO PLANO — PID',process.pid)
print('Vídeo:',video)
print('Intervalo:',INICIO_S,'a',FIN_S if FIN_S is not None else 'final del vídeo')
print('Progreso:',logpath)
print('Resultados:',BASE/'resultados')
print('La celda puede terminar mientras el análisis continúa. El servidor debe permanecer encendido.')
print('La primera vez descarga dependencias y pesos. Lanzado NO significa terminado.')


LANZADO EN SEGUNDO PLANO — PID 4447
Vídeo: /workspace/equipo5/NOA/consejo_131026.mp4
Intervalo: 0.0 a final del vídeo
Progreso: /workspace/equipo5/PAULA/consejo_integrado/ejecucion.log
Resultados: /workspace/equipo5/PAULA/consejo_integrado/resultados
La celda puede terminar mientras el análisis continúa. El servidor debe permanecer encendido.
La primera vez descarga dependencias y pesos. Lanzado NO significa terminado.


In [3]:
from pathlib import Path
import json

base = Path("/workspace/equipo5/PAULA/consejo_integrado")
estado = base / "resultados/estado.json"

if estado.exists():
    print("ESTADO:", json.loads(estado.read_text()).get("estado"))

log = base / "ejecucion.log"
print("\nÚLTIMOS MENSAJES:\n")
print(log.read_text(encoding="utf-8", errors="replace")[-4000:])

ESTADO: completo

ÚLTIMOS MENSAJES:

de cálculo
Plano 21/65: 1402.27–1411.63 s
Guardado plano 21; 4.0 s de cálculo
Plano 22/65: 1411.63–1577.47 s
  Plano 22: segundo 1492.1 / 1577.5
  Plano 22: segundo 1574.1 / 1577.5
Guardado plano 22; 61.3 s de cálculo
Plano 23/65: 1577.47–1605.67 s
Guardado plano 23; 12.2 s de cálculo
Plano 24/65: 1605.67–1629.83 s
Guardado plano 24; 8.8 s de cálculo
Plano 25/65: 1629.83–1634.37 s
Guardado plano 25; 2.2 s de cálculo
Plano 26/65: 1634.37–1652.77 s
Guardado plano 26; 7.1 s de cálculo
Plano 27/65: 1652.77–1658.87 s
Guardado plano 27; 2.9 s de cálculo
Plano 28/65: 1658.87–1714.30 s
Guardado plano 28; 21.0 s de cálculo
Plano 29/65: 1714.30–1732.57 s
Guardado plano 29; 7.9 s de cálculo
Plano 30/65: 1732.57–1777.67 s
Guardado plano 30; 16.0 s de cálculo
Plano 31/65: 1777.67–1784.30 s
Guardado plano 31; 3.1 s de cálculo
Plano 32/65: 1784.30–1809.43 s
Guardado plano 32; 9.1 s de cálculo
Plano 33/65: 1809.43–1817.03 s
Guardado plano 33; 3.6 s de cálculo
Plano